In [ ]:
#|default_exp kucoin

In [ ]:
#|hide
%load_ext autoreload
%autoreload 2

## KuCoin

Manages the KuCoin data download and processing using the [ccxt](https://github.com/ccxt/ccxt) package.
Candles (OHLCV) are downloaded through `ccxt.kucoin` and saved, one file per trading pair, into a
folder named after the exchange (e.g. `../data/kucoin`).

Notes:

- KuCoin serves deep history (up to 1500 candles per request), so full backfills are possible.
- Rate limits are handled by ccxt (`enableRateLimit=True`).

** Finally, datetime columns are in UTC. **

In [ ]:
#| export

import os
import time
import datetime as dt
from datetime import datetime
import pandas as pd
import ccxt

In [ ]:
#| export

def retry_fetch_ohlcv(exchange, max_retries, symbol, timeframe, since, limit, verbose=False):
    """
    Fetch a single page of OHLCV candles from an exchange, retrying on failure.

    Args:
        exchange (ccxt.Exchange): Instantiated ccxt exchange object
        max_retries (int): Maximum number of retries before raising the last error
        symbol (str): Trading pair symbol (e.g. "BTC/USDT")
        timeframe (str): Candle timeframe (e.g. "1m", "1h", "1d")
        since (int): Start time in milliseconds since epoch (UTC)
        limit (int): Maximum number of candles to fetch
        verbose (bool, optional): If True, prints progress messages. Defaults to False

    Returns:
        list: List of OHLCV candles [timestamp_ms, open, high, low, close, volume]

    Raises:
        Exception: The last ccxt error if all retries fail
    """
    num_retries = 0
    while num_retries <= max_retries:
        try:
            ohlcv = exchange.fetch_ohlcv(symbol=symbol, timeframe=timeframe, since=since, limit=limit)
            if verbose and len(ohlcv):
                print(f"Fetched {len(ohlcv)} {symbol} candles from "
                      f"{exchange.iso8601(ohlcv[0][0])} to {exchange.iso8601(ohlcv[-1][0])}")
            time.sleep(exchange.rateLimit / 1000)
            return ohlcv
        except Exception as error:
            num_retries += 1
            if num_retries > max_retries:
                raise
            print(f"Error fetching {symbol}: {error}. Retry {num_retries}/{max_retries}")
            time.sleep(2)

In [ ]:
#| export

def scrape_ohlcv(exchange, symbol, timeframe, since, end=None, max_retries=3, limit=1500, verbose=False):
    """
    Download OHLCV candles in pages of `limit` bars between `since` and `end`.

    Args:
        exchange (ccxt.Exchange): Instantiated ccxt exchange object (markets loaded)
        symbol (str): Trading pair symbol (e.g. "BTC/USDT")
        timeframe (str): Candle timeframe (e.g. "1m", "1h", "1d")
        since (int or str): Start time in milliseconds since epoch or ISO 8601 string
        end (int or str, optional): End time in milliseconds or ISO 8601 string.
            Defaults to now.
        max_retries (int, optional): Retries per page. Defaults to 3
        limit (int, optional): Candles per request. Defaults to 1500 (KuCoin maximum)
        verbose (bool, optional): If True, prints progress messages. Defaults to False

    Returns:
        list: List of OHLCV candles [timestamp_ms, open, high, low, close, volume]
    """
    if isinstance(since, str):
        since = exchange.parse8601(since)
    now = exchange.milliseconds()
    if end is None:
        end = now
    elif isinstance(end, str):
        end = exchange.parse8601(end)
    tf_ms = exchange.parse_timeframe(timeframe) * 1000
    all_ohlcv = []
    fetch_since = since
    remain_bars = (end - fetch_since) / tf_ms + 1  # +1 to include the end candle
    while fetch_since < min(end, now) and remain_bars > 0:
        batch = int(min(limit, remain_bars))
        if batch < 1:
            break
        ohlcv = retry_fetch_ohlcv(exchange, max_retries, symbol, timeframe, fetch_since, batch, verbose=verbose)
        all_ohlcv = all_ohlcv + ohlcv
        fetch_since = fetch_since + batch * tf_ms
        remain_bars = remain_bars - batch
        if not ohlcv:
            break
    return exchange.filter_by_since_limit(all_ohlcv, since, None, key=0)

In [ ]:
#| export

def ohlcv_to_df(ohlcv, symbol):
    """
    Convert a raw ccxt OHLCV list into a tidy DataFrame.

    Args:
        ohlcv (list): List of candles [timestamp_ms, open, high, low, close, volume]
        symbol (str): Trading pair symbol added as the `pair` column

    Returns:
        pandas.DataFrame: DataFrame with columns:
            - datetime: Candle timestamp (UTC, timezone-aware)
            - open, high, low, close, volume: OHLCV values
            - pair: Trading pair symbol
        Sorted by datetime with duplicate timestamps removed.
    """
    df = pd.DataFrame(ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
    df['datetime'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
    df['pair'] = symbol
    df = df.drop(columns=['timestamp'])
    df = df[['datetime', 'open', 'high', 'low', 'close', 'volume', 'pair']]
    df = df.drop_duplicates(subset='datetime').sort_values('datetime').reset_index(drop=True)
    return df

In [ ]:
#| export

def kucoin_ohlcv(symbol='BTC/USDT', timeframe='1h', since=None, end=None,
                 max_retries=3, limit=1500, exchange=None, verbose=False):
    """
    Download OHLCV candles for a symbol from KuCoin.

    Args:
        symbol (str, optional): Trading pair symbol. Defaults to "BTC/USDT"
        timeframe (str, optional): Candle timeframe. Defaults to "1h"
        since (int or str, optional): Start time (ms since epoch or ISO 8601).
            If None, downloads the most recent ~`limit` candles.
        end (int or str, optional): End time (ms or ISO 8601). Defaults to now
        max_retries (int, optional): Retries per page. Defaults to 3
        limit (int, optional): Candles per request. Defaults to 1500
        exchange (ccxt.kucoin, optional): Reusable exchange instance. If None, a new one is created
        verbose (bool, optional): If True, prints progress messages. Defaults to False

    Returns:
        pandas.DataFrame: Tidy OHLCV DataFrame (see `ohlcv_to_df`)
    """
    if exchange is None:
        exchange = ccxt.kucoin({'enableRateLimit': True})
        exchange.load_markets()
    if since is None:
        tf_ms = exchange.parse_timeframe(timeframe) * 1000
        since = exchange.milliseconds() - (limit - 20) * tf_ms
    ohlcv = scrape_ohlcv(exchange, symbol, timeframe, since, end=end,
                         max_retries=max_retries, limit=limit, verbose=verbose)
    return ohlcv_to_df(ohlcv, symbol)

In [ ]:
#| export

def kucoin_usdt_tokens(exchange=None):
    """
    Retrieves all active KuCoin spot trading pairs quoted in USDT.

    Args:
        exchange (ccxt.kucoin, optional): Reusable exchange instance. If None, a new one is created

    Returns:
        pandas.DataFrame: DataFrame with columns:
            - id: KuCoin market id (e.g. 'BTC-USDT')
            - symbol: Unified ccxt symbol (e.g. 'BTC/USDT')
            - base: Base currency (e.g. 'BTC')
            - quote: Always 'USDT' for this filtered dataset
            - active: Whether the market is currently active
    """
    if exchange is None:
        exchange = ccxt.kucoin({'enableRateLimit': True})
    markets = exchange.load_markets()
    rows = [{'id': m['id'], 'symbol': m['symbol'], 'base': m['base'],
             'quote': m['quote'], 'active': m.get('active', True)}
            for m in markets.values() if m.get('spot', False) and m['quote'] == 'USDT']
    return pd.DataFrame(rows)

#### Example / tests

In [ ]:
#|eval: false
# Quick live test: download ~2 days of hourly BTC/USDT candles
start = (pd.Timestamp.now(tz='UTC') - pd.Timedelta(days=2)).strftime('%Y-%m-%dT%H:%M:%SZ')
df_test = kucoin_ohlcv(symbol='BTC/USDT', timeframe='1h', since=start)
assert not df_test.empty
assert list(df_test.columns) == ['datetime', 'open', 'high', 'low', 'close', 'volume', 'pair']
assert df_test['datetime'].dt.tz is not None
assert df_test['datetime'].is_monotonic_increasing
assert (df_test['pair'] == 'BTC/USDT').all()
assert len(df_test) > 24
df_test.tail()

,datetime,open,high,low,close,volume,pair
43,2026-07-08 09:00:00+00:00,62022.0,62166.0,61746.0,62055.4,88.046372,BTC/USDT
44,2026-07-08 10:00:00+00:00,62055.4,62192.5,61903.8,62182.7,54.975636,BTC/USDT
45,2026-07-08 11:00:00+00:00,62182.7,62395.5,62053.8,62299.9,52.655781,BTC/USDT
46,2026-07-08 12:00:00+00:00,62299.9,62425.0,61964.3,61964.4,60.474317,BTC/USDT
47,2026-07-08 13:00:00+00:00,61964.4,62000.2,61848.1,61881.8,29.330686,BTC/USDT


In [ ]:
#|eval: false
# Offline test: ohlcv_to_df shapes raw candles correctly and removes duplicates
sample = [[1700000000000, 1.0, 2.0, 0.5, 1.5, 10.0],
          [1700003600000, 1.5, 2.5, 1.0, 2.0, 20.0],
          [1700003600000, 1.5, 2.5, 1.0, 2.0, 20.0]]  # duplicate on purpose
df_sample = ohlcv_to_df(sample, 'TEST/USDT')
assert len(df_sample) == 2
assert list(df_sample.columns) == ['datetime', 'open', 'high', 'low', 'close', 'volume', 'pair']
assert str(df_sample['datetime'].dt.tz) == 'UTC'
df_sample

,datetime,open,high,low,close,volume,pair
0,2023-11-14 22:13:20+00:00,1.0,2.0,0.5,1.5,10.0,TEST/USDT
1,2023-11-14 23:13:20+00:00,1.5,2.5,1.0,2.0,20.0,TEST/USDT


In [ ]:
#|eval: false
# Live test: token universe contains the major pairs
tokens = kucoin_usdt_tokens()
assert not tokens.empty
assert 'BTC/USDT' in tokens['symbol'].tolist()
assert (tokens['quote'] == 'USDT').all()
tokens.head()

,id,symbol,base,quote,active
0,AVA-USDT,AVA/USDT,AVA,USDT,True
1,MTV-USDT,MTV/USDT,MTV,USDT,True
2,TEL-USDT,TEL/USDT,TEL,USDT,True
3,TT-USDT,TT/USDT,TT,USDT,True
4,XMR-USDT,XMR/USDT,XMR,USDT,True


In [ ]:
#| export

def save_file(df, folder_path, file_name, type="parquet"):
    """
    Save a pandas DataFrame to a file in either CSV or Parquet format.

    Args:
        df (pandas.DataFrame): The DataFrame to save
        folder_path (str): Directory path where the file will be saved
        file_name (str): Name of the file without extension
        type (str, optional): File format - either "csv" or "parquet". Defaults to "parquet"

    The function saves the DataFrame to the specified path, handling the file extension automatically.
    For CSV files, the index is not saved. Creates the folder if it doesn't exist.
    """
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
    if type == "csv":
        df.to_csv(f"{folder_path}/{file_name}.csv", index=False)
    elif type == "parquet":
        df.to_parquet(f"{folder_path}/{file_name}.parquet")
    else:
        raise ValueError(f"Type {type} not supported. Use 'csv' or 'parquet'")

In [ ]:
#| export

def symbol_to_file_name(symbol, timeframe='1h'):
    """
    Convert a ccxt symbol and timeframe into a file name (without extension).

    Example: ('BTC/USDT', '1h') -> 'BTC-USDT_1h'
    """
    return f"{symbol.replace('/', '-')}_{timeframe}"

def file_name_to_symbol(file_name):
    """
    Convert a file name back into a ccxt symbol.

    Example: 'BTC-USDT_1h.parquet' -> 'BTC/USDT'
    """
    base = os.path.splitext(file_name)[0]
    pair = base.rsplit('_', 1)[0]
    return pair.replace('-', '/')

In [ ]:
#|eval: false
# Offline test: save_file round-trip and file-name helpers
import tempfile
tmp_dir = tempfile.mkdtemp()
save_file(df_sample, tmp_dir, 'TEST-USDT_1h', type='parquet')
df_back = pd.read_parquet(f"{tmp_dir}/TEST-USDT_1h.parquet")
assert df_back.shape == df_sample.shape
assert list(df_back.columns) == list(df_sample.columns)
assert symbol_to_file_name('BTC/USDT', '1h') == 'BTC-USDT_1h'
assert file_name_to_symbol('BTC-USDT_1h.parquet') == 'BTC/USDT'
print('save_file round-trip OK')

save_file round-trip OK


In [ ]:
#| export

def kucoin_to_file(folder_path="../data/kucoin", token_list=['BTC/USDT', 'ETH/USDT'], type="parquet",
                   timeframe='1h', refresh_hours=24, first_date='2021-01-01T00:00:00Z',
                   all_tokens=True, pause=1, verbose=False):
    """
    Downloads and maintains historical KuCoin OHLCV data, saving one file per pair.

    Args:
        folder_path (str): Path where pair data files will be stored, named after the
            exchange. Defaults to "../data/kucoin"
        token_list (list): List of ccxt symbols to process. Defaults to ['BTC/USDT', 'ETH/USDT']
        type (str): File format to save data - either "csv" or "parquet". Defaults to "parquet"
        timeframe (str): Candle timeframe (e.g. "1m", "1h", "1d"). Defaults to "1h"
        refresh_hours (int): Hours of the most recent data to re-download when updating.
            Defaults to 24
        first_date (str): ISO 8601 start date used for the initial full-history download
            of a pair. Defaults to '2021-01-01T00:00:00Z'
        all_tokens (bool): If True, includes any additional pairs found in the folder path.
            Defaults to True
        pause (int): Seconds to wait between pairs. Defaults to 1
        verbose (bool): If True, prints download progress. Defaults to False

    The function:
    - Creates the folder_path if it doesn't exist
    - Date/Time is UTC
    - For each pair, checks if a data file exists:
        - If exists: Loads the file and appends new data, refreshing the last `refresh_hours`
        - If not exists: Downloads full history starting from `first_date`
    - Saves data in the specified format, handling duplicates and sorting by datetime
    """
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
    elif all_tokens:
        file_names = [f for f in os.listdir(folder_path)
                      if f.endswith(type) and f"_{timeframe}." in f]
        tokens_in_folder = [file_name_to_symbol(f) for f in file_names]
        token_list = list(set(token_list + tokens_in_folder))
    exchange = ccxt.kucoin({'enableRateLimit': True})
    exchange.load_markets()
    for token in token_list:
        print(f"Processing {token}")
        file_name = symbol_to_file_name(token, timeframe)
        file_path = f"{folder_path}/{file_name}.{type}"
        try:
            if os.path.exists(file_path):
                if type == "csv":
                    df = pd.read_csv(file_path)
                    df['datetime'] = pd.to_datetime(df['datetime'], utc=True)
                elif type == "parquet":
                    df = pd.read_parquet(file_path)
                else:
                    raise ValueError(f"Type {type} not supported")
                last_date = pd.to_datetime(df['datetime'].max(), utc=True)
                since = last_date - pd.Timedelta(hours=refresh_hours)
                df = df[pd.to_datetime(df['datetime'], utc=True) < since]
                df_new = kucoin_ohlcv(symbol=token, timeframe=timeframe,
                                      since=since.strftime('%Y-%m-%dT%H:%M:%SZ'),
                                      exchange=exchange, verbose=verbose)
                df = pd.concat([df, df_new])
                df = df.drop_duplicates(subset='datetime').sort_values('datetime').reset_index(drop=True)
            else:
                df = kucoin_ohlcv(symbol=token, timeframe=timeframe, since=first_date,
                                  exchange=exchange, verbose=verbose)
            if not df.empty:
                save_file(df, folder_path, file_name, type)
        except Exception as e:
            print(f"Error processing {token}: {e}")
        if pause > 0:
            time.sleep(pause)

#### Example / tests

In [ ]:
#|eval: false
# Live test: download BTC/USDT into a temporary exchange folder, then re-run incrementally
import tempfile
kucoin_dir = tempfile.mkdtemp()
recent = (pd.Timestamp.now(tz='UTC') - pd.Timedelta(days=3)).strftime('%Y-%m-%dT%H:%M:%SZ')
kucoin_to_file(folder_path=kucoin_dir, token_list=['BTC/USDT'], type='parquet',
               timeframe='1h', first_date=recent, pause=0)
assert os.path.exists(f"{kucoin_dir}/BTC-USDT_1h.parquet")
df1 = pd.read_parquet(f"{kucoin_dir}/BTC-USDT_1h.parquet")
n1 = len(df1)
assert n1 > 0
# Incremental re-run must not shrink the file and must not create duplicates
kucoin_to_file(folder_path=kucoin_dir, token_list=['BTC/USDT'], type='parquet',
               timeframe='1h', first_date=recent, pause=0)
df2 = pd.read_parquet(f"{kucoin_dir}/BTC-USDT_1h.parquet")
assert len(df2) >= n1
assert not df2['datetime'].duplicated().any()
print(f"First run: {n1} rows, second run: {len(df2)} rows")

Processing BTC/USDT
Processing BTC/USDT
First run: 72 rows, second run: 72 rows


In [ ]:
#|eval: false
# Download / update a set of USDT pairs into the exchange-named data folder
# For all USDT pairs use: token_list = kucoin_usdt_tokens()['symbol'].tolist()
kucoin_to_file(folder_path="../data/kucoin",
               token_list=['BTC/USDT', 'ETH/USDT', 'SOL/USDT','XMR/USDT'],
               type="parquet", timeframe='1h')

Processing ETH/USDT
Processing XMR/USDT
Processing BTC/USDT
Processing SOL/USDT


In [ ]:
#|hide
import nbdev; nbdev.nbdev_export()